Migrated from Databricks to Neuron (auto-port).

In [ ]:
import os
import sys
from pathlib import Path

for _candidate in (
    "/home/jovyan/work/default/rashmi-ranjan/lib",
    str(Path.cwd().parent / "lib"),
    str(Path.cwd() / "lib"),
):
    if _candidate not in sys.path and Path(_candidate).is_dir():
        sys.path.insert(0, _candidate)

from neuron_runtime import (
    GoogleSheetReader,
    SlackReportsTrigger,
    get_sql_df,
    load_service_account,
    load_slack_token,
    make_unique_columns,
)

service_account = load_service_account()
slack_token = load_slack_token()

In [ ]:
# DBTITLE 1,Cell 1
# MAGIC
# MAGIC import trino
# MAGIC import pandas as pd
# MAGIC
# MAGIC query = """
# MAGIC WITH welcome_calls AS (
# MAGIC     SELECT
# MAGIC         json_parse(t.meta_data) AS call_meta,
# MAGIC         try(date_parse(
# MAGIC             json_extract_scalar(json_parse(t.meta_data), '$.callInitiationTime'),
# MAGIC             '%Y/%m/%d %H:%i:%s +0530'
# MAGIC         )) AS call_timestamp
# MAGIC     FROM awsdatacatalog.crm_tickets.tickets t
# MAGIC     WHERE t.ticket_group = 'GI_WELCOME_CALLING'
# MAGIC       AND json_extract_scalar(json_parse(t.meta_data), '$.userId') = 'gousiya.gb@navi.com'
# MAGIC )
# MAGIC SELECT
# MAGIC     CAST(call_timestamp AT TIME ZONE 'Asia/Kolkata' AS DATE) AS Call_Date,
# MAGIC     CASE
# MAGIC         WHEN day_of_week(CAST(call_timestamp AT TIME ZONE 'Asia/Kolkata' AS DATE)) = 6
# MAGIC             THEN date_add(
# MAGIC                 'day',
# MAGIC                 2,
# MAGIC                 CAST(call_timestamp AT TIME ZONE 'Asia/Kolkata' AS DATE)
# MAGIC             )   -- Saturday call → audit on Monday
# MAGIC         ELSE date_add(
# MAGIC                 'day',
# MAGIC                 1,
# MAGIC                 CAST(call_timestamp AT TIME ZONE 'Asia/Kolkata' AS DATE)
# MAGIC             )   -- All other days → next day
# MAGIC     END AS Audit_date,
# MAGIC     COUNT(DISTINCT json_extract_scalar(call_meta, '$.callId')) AS Total_Calls
# MAGIC FROM welcome_calls
# MAGIC WHERE call_timestamp IS NOT NULL
# MAGIC   AND CAST(call_timestamp AT TIME ZONE 'Asia/Kolkata' AS DATE)
# MAGIC       BETWEEN date_add('day', -90, CAST(at_timezone(current_timestamp, 'Asia/Kolkata') AS DATE))
# MAGIC           AND date_add('day', -1, CAST(at_timezone(current_timestamp, 'Asia/Kolkata') AS DATE))
# MAGIC GROUP BY 1, 2
# MAGIC ORDER BY 1 DESC
# MAGIC """
# MAGIC
# MAGIC df  = get_sql_df(query)
# MAGIC if not df.empty:
# MAGIC     display(df)

In [ ]:
# Python cell
import os
import shutil

if not os.path.exists("data"):
    os.makedirs("data")

gsheet_url = "https://docs.google.com/spreadsheets/d/1mu6xk-ZqkxUb1egHfo_nokzhemjTVp_aPlpIzzgq-vg/edit?gid=1975214994#gid=1975214994"
worksheet_name = "Connected Call Count New"
gs_reader = GoogleSheetReader(gsheet_url, service_account)

gs_reader.clear_sheets(
    worksheet_name,
    start_row=1,
    start_column=1
)

combined_df = pd.concat([df], ignore_index=True) 

gs_reader.set_df_in_sheets( 
worksheet_name, 
combined_df, 
include_headers=True, 
row=1, 
col=1
)